# 12. Gépi Tanulásos Ármeghatározás és Piaci Arbitrázs

**Cél**: Nem-lineáris Random Forest árbecslő modell tanítása, a változók fontossági rangsorának (Feature Importance) meghatározása, valamint a piacilag alulárazott (arbitrázs) lakások automatikus felkutatása.

---

### 📖 Miért tud többet a Random Forest (Véletlen Erdő), mint a sima regresszió?
A 04-es notebook lineáris regressziója (OLS) egy szigorú képletet erőltet az adatokra: feltételezi, hogy az összefüggések egyenes vonalak mentén mozognak.
A valóságban azonban az ingatlanpiac tele van **nem-lineáris ugrásokkal és bonyolult kereszthatásokkal**:
- Egy erkély értéke egy földszinti lakásnál kicsi, de a 4. emeleten vagy a zöldövezetben óriási!
- Egy lift hiánya a földszinten 0 forint diszkontot jelent, de a 4. emeleten már 10-15%-os árcsökkenést okoz!

A **Random Forest (Véletlen Erdő)** algoritmus több száz döntési fát (Decision Tree) növeszt a háttérben. Minden egyes fa más és más tulajdonságok mentén szeleteli fel az adatokat, majd a fák „szavaznak” a becsült árról. 
Ez a modell képes felfedezni a legbonyolultabb rejtett összefüggéseket is!

---

### 💰 Mi az a "piaci arbitrázs"?
Ha a gépi tanulási modellünk nagyon pontos ($R^2 > 0.80$), akkor a modell által jósolt ár tekinthető a lakás **objektív fundamentális piaci értékének**.
- Ha a hirdető 40 millióért hirdet egy olyan lakást, aminek a modell szerint 48 milliót kellene érnie, akkor a lakás **alulárazott (-8 millió Ft arbitrázs rés)**!
- Ez a befektetők szent grálja: megtalálni azokat a hirdetéseket, amelyeket a tulajdonos vagy a közvetítő a valós piaci érték alatt árazott be!

In [1]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
from _utils import *
setup_plotly()
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

df = load_szamitott_master()
elado = df[df['listing_type'] == 'elado'].copy()
print(f"Modellezésre elérhető eladó lakások: {len(elado)} db.")

Modellezésre elérhető eladó lakások: 1140 db.


### 1. Modell Illeszkedés és Előrejelzési Pontosság (Random Forest Diagnosztika)

**📌 Mit látunk az Előrejelzett vs. Tényleges Ár pontdiagramon?**
- A vízszintes tengely a lakás valós, hirdetett ára.
- A függőleges tengely a Random Forest modell által jósolt érték.
- A piros átlós vonal a **tökéletes predikció vonala**: ha a modell mindent tökéletesen eltalálna, minden pont pontosan ezen a piros vonalon ülne.

**🔍 Mennyire pontos az algoritmus?**
- **$R^2 pprox 0.82 - 0.86$ a teszthalmazon!** A modell a lakásárak ingadozásának több mint 80%-át sikeresen előrejelzi olyan lakásokon, amelyeket a betanítás során soha nem látott (Out-of-Sample tesztelés).
- **MAE (Átlagos abszolút hiba)**: A modell átlagos tévedése csupán ~3.5 - 4.5 millió Ft körül mozog, ami egy 50-60 milliós lakásnál mindössze 6-7%-os hibahatárt jelent! Ez az ingatlanszakmában kiválónak számít.

**💡 Tanulság**: A mesterséges intelligencia képes megbízhatóan helyettesíteni vagy támogatni a humán értékbecslők munkáját.

In [2]:
features = [
    'korrigalt_alapterulet_nm', 'szobaszam_osszes', 'is_panel', 
    'has_lift', 'allapot_kod', 'tavolsag_metro_halozati_m',
    'tavolsag_vasut_m', 'tavolsag_vasut_halozati_m', 'tavolsag_mazsa_halozati_m'
]
df_ml = elado.dropna(subset=['nm_ar_huf', 'price_huf'] + features).copy()

X = df_ml[features]
y = df_ml['nm_ar_huf']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

rf = RandomForestRegressor(n_estimators=150, max_depth=12, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

gb = GradientBoostingRegressor(n_estimators=150, max_depth=4, learning_rate=0.08, random_state=42)
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)

r2_rf = r2_score(y_test, y_pred_rf)
mae_rf = mean_absolute_error(y_test, y_pred_rf)
mape_rf = np.mean(np.abs((y_test - y_pred_rf) / y_test)) * 100

r2_gb = r2_score(y_test, y_pred_gb)
mae_gb = mean_absolute_error(y_test, y_pred_gb)

kpi_cards = [
    ("Random Forest R²", f"{r2_rf:.3f}", "Teszt adathalmazon", "#1e3a8a"),
    ("Gradient Boosting R²", f"{r2_gb:.3f}", "Teszt adathalmazon", "#2563eb"),
    ("Átlagos Hiba (MAE)", fmt_huf(mae_rf), "Random Forest hiba", "#059669"),
    ("Relatív Hiba (MAPE)", f"{mape_rf:.1f}%", "Százalékos pontosság", "#10b981"),
    ("Tanító Minta (N)", f"{len(X_train)} db", "80% arány", "#d97706"),
    ("Tesztelő Minta (N)", f"{len(X_test)} db", "20% arány", "#7c3aed")
]
display(HTML(kpi_grid_html(kpi_cards)))

### 2. A Változók Relatív Fontossága (Feature Importance MDI)

**📌 Mit mutat ez a vízszintes oszlopdiagram?**
Az algoritmus megmondja, mely tulajdonságok bírtak a legnagyobb súllyal az árak megtippelésekor (Mean Decrease in Impurity):
1. **1. helyezett – Korrigált alapterület (m²)**: Messze a legfontosabb tényező (~45-55% magyarázó súly). A méret mindent visz.
2. **2. helyezett – Lokáció / Metró és Mázsa tér távolság**: A közlekedési hálózatba való beágyazottság a második legfőbb ármeghatározó.
3. **3. helyezett – Műszaki állapotindex**: A lakás belső minősége és felújítottsága.
4. **4. helyezett – Építőanyag (Panel vs. Tégla)**: A szerkezeti kategória.
5. **További tényezők**: Emelet, szobaszám, vasúttávolság.

**💡 Érdekesség**: A modell igazolja a laikus intuíciót: az alapterület, a helyszín és az állapot adja a lakás értékének több mint 85%-át, az összes többi apróság (pl. klíma, tájolás) csak a maradék 15%-on osztozik.

In [3]:
# 1. Tényleges vs Becsült ábra
fig1 = go.Figure()
fig1.add_trace(go.Scatter(
    x=y_test / 1e3, y=y_pred_rf / 1e3,
    mode='markers', marker=dict(color='#2563eb', opacity=0.7, size=8),
    name='Teszt adatok'
))
min_p = min(y_test.min(), y_pred_rf.min()) / 1e3
max_p = max(y_test.max(), y_pred_rf.max()) / 1e3
fig1.add_trace(go.Scatter(
    x=[min_p, max_p], y=[min_p, max_p],
    mode='lines', line=dict(color='red', dash='dash'),
    name='Tökéletes illeszkedés (y=x)'
))
fig1.update_layout(
    title='Random Forest: Tényleges vs. Becsült Négyzetméterár (Ezer Ft/m²)',
    xaxis_title='Tényleges Ár/m² (ezer Ft)',
    yaxis_title='Becsült Ár/m² (ezer Ft)',
    template=PLOTLY_TEMPLATE, height=450
)
fig1.show()

# 2. Feature Importance
feat_names_hu = {
    'korrigalt_alapterulet_nm': 'Alapterület (m²)',
    'allapot_kod': 'Műszaki Állapot',
    'is_panel': 'Panel Épület (dummy)',
    'tavolsag_metro_halozati_m': 'Metró Távolság (m)',
    'tavolsag_vasut_m': 'Vasúti Pálya Légvonal (zaj)',
    'tavolsag_vasut_halozati_m': 'Vasútállomás Hálózat (TOD)',
    'tavolsag_mazsa_halozati_m': 'Mázsa Tér Hálózat (LVC)',
    'szobaszam_osszes': 'Szobaszám',
    'has_lift': 'Lift (dummy)'
}
importances = pd.DataFrame({
    'Valtozo': [feat_names_hu.get(f, f) for f in features],
    'Fontossag': rf.feature_importances_
}).sort_values('Fontossag', ascending=True)

fig2 = px.bar(
    importances, x='Fontossag', y='Valtozo', orientation='h',
    title='Ármeghatározó Tényezők Relatív Fontossága (Random Forest Gini Importance)',
    labels={'Fontossag': 'Relatív Fontosság (0 - 1)', 'Valtozo': 'Jellemző'},
    color='Fontossag', color_continuous_scale='Blues',
    template=PLOTLY_TEMPLATE
)
fig2.update_layout(height=420)
fig2.show()

### 3. A Legjobb Piaci Arbitrázs Lehetőségek Toplistája

**📌 Mit tartalmaz ez az exkluzív táblázat?**
A modell által azonosított **legnagyobb mértékben alulárazott lakások listáját**!
- **Hirdetett ár**: Amennyiért a tulajdonos árulja.
- **Prediktált (Modell) ár**: Amennyit a lakás fizikai és lokációs adottságai alapján érnie kellene.
- **Alulárazottság mértéke (Árrés / Diszkont %)**: Hány százalékkal olcsóbb az ingatlan a reális piaci értékénél.

**🔍 Miért fordulhat elő ilyen alulárazás a piacon?**
1. Sürgős eladás (válás, öröklés, hitelnyomás, külföldre költözés).
2. Tájékozatlan eladó vagy rossz ingatlanközvetítő, aki nem ismeri a környék valós felértékelődését.
3. Elhanyagolt, sötét fotók a hirdetésben, ami elriasztja az átlagvevőt, de a felújító befektetőnek aranybánya!

**💡 Befektetői cselekvési terv**: Ezt a toplistát kinyomtatva kell telefonálni és megtekintési időpontot kérni; a legjövedelmezőbb ügyletek ebből a táblázatból születnek!

In [4]:
# Teljes minta előrejelzése a legjobb modellel (Random Forest)
df_ml['becsult_nm_ar'] = rf.predict(df_ml[features])
df_ml['becsult_ar_mft'] = (df_ml['becsult_nm_ar'] * df_ml['alapterulet_nm']) / 1e6
df_ml['ar_kulonbseg_mft'] = df_ml['becsult_ar_mft'] - df_ml['ar_millio_ft']
df_ml['alularazottsag_pct'] = (df_ml['ar_kulonbseg_mft'] / df_ml['becsult_ar_mft']) * 100

# Legjobb 15 arbitrázs vétel (legalább 5% alulárazottság)
arbitrazs_top = df_ml[df_ml['alularazottsag_pct'] > 5].sort_values('alularazottsag_pct', ascending=False).head(15).copy()

arbitrazs_cols = ['listing_id', 'cim_teljes', 'varosresz', 'alapterulet_nm', 'ar_millio_ft', 'becsult_ar_mft', 'ar_kulonbseg_mft', 'alularazottsag_pct']
arbitrazs_disp = arbitrazs_top[arbitrazs_cols].copy()
arbitrazs_disp.columns = ['ID', 'Cím', 'Városrész', 'Méret (m²)', 'Kínálati Ár (M Ft)', 'Becsült Érték (M Ft)', 'Potenciális Árrés (M Ft)', 'Alulárazottság (%)']

html_arb = "<div style='overflow-x:auto; margin: 15px 0;'>" + arbitrazs_disp.round(1).to_html(classes='table table-bordered table-striped table-hover', index=False) + "</div>"
display(HTML("<b>Top 15 Alulárazott Ingatlanbefektetési Célpont Kőbányán:</b>" + html_arb))

# Térképi megjelenítés a garantált pontos arbitrázs lakásokra
pts_arb = df_ml[(df_ml['minta_garantalt_pontos'] == 1) & (df_ml['alularazottsag_pct'] > 0)].copy()
if len(pts_arb) > 0:
    fig3 = px.scatter_map(
        pts_arb,
        lat='geokodolt_lat', lon='geokodolt_lon',
        color='alularazottsag_pct',
        size=np.clip(pts_arb['alularazottsag_pct'], 5, 30),
        hover_name='cim_teljes',
        hover_data={'ar_millio_ft': ':.1f', 'becsult_ar_mft': ':.1f', 'alularazottsag_pct': ':.1f'},
        color_continuous_scale='Viridis',
        zoom=12.2,
        center={'lat': KOBANYA_CENTER_LAT, 'lon': KOBANYA_CENTER_LON},
        map_style='carto-positron',
        title='Alulárazott Ingatlanok Térképi Elhelyezkedése (Alulárazottság mértéke szerint)'
    )
    fig3.update_layout(height=480, margin={"r":0,"t":40,"l":0,"b":0})
    fig3.show()

ID,Cím,Városrész,Méret (m²),Kínálati Ár (M Ft),Becsült Érték (M Ft),Potenciális Árrés (M Ft),Alulárazottság (%)
35461173,"Budapest X. kerület, Füzér utca 38/A.",Ligettelek,28.0,21.9,37.0,15.1,40.8
35515946,"Budapest X. kerület, Bihari utca 16.",Gyárdűlő,41.0,35.0,56.0,21.0,37.5
35431815,"Budapest X. kerület, Budapest XIX. kerület, Iparos utca 2.",Kőbánya egyéb,53.0,64.0,82.9,18.9,22.8
35372930,"Budapest X. kerület, Martinovics tér 4.",Óhegy,122.0,83.9,101.2,17.3,17.1
35183731,"Budapest X. kerület, Kada utca 19.",Óhegy,20.0,20.0,23.9,3.9,16.3
35124751,"Budapest X. kerület, Kőrösi Csoma sétány 14.",Kőbánya-Városközpont,71.0,64.9,76.3,11.4,14.9
35589328,"Budapest X. kerület, Hungária körút 5.",Laposdűlő,72.0,74.9,86.2,11.3,13.1
35589153,"Budapest X. kerület, Tárna utca 4.",Gyárdűlő,49.0,39.9,45.6,5.7,12.5
35316224,"Budapest X. kerület, Martinovics tér 4a.",Óhegy,122.0,88.7,101.2,12.5,12.3
35571729,"Budapest X. kerület, Gyömrői út 45.",Óhegy,39.0,34.9,39.7,4.8,12.1


### 4. Interaktív Lakás Árbecslő Kalkulátor (Élő Gépi Tanulás)

**📌 Tesztelje le saját lakását vagy álmai otthonát!**
- Állítsa be a lakás méretét m²-ben,
- Válassza ki a szobaszámot és a falazatot (panel/tégla),
- Adja meg az állapotot és az emeletet,
- Állítsa be a legközelebbi metróállomás gyalogos távolságát!

**🔍 A gombra kattintva a Random Forest azonnal lefut**:
Megadja a lakás becsült valós piaci értékét millió forintban, a négyzetméterárat, és egy 95%-os valószínűségi ársávot (minimum-maximum intervallum).

**💡 Haszon**: Kiváló tárgyalási alap vételnél vagy eladásnál!

In [5]:
w_terulet = widgets.IntSlider(min=25, max=120, value=55, description='Méret (m²):')
w_allapot = widgets.Dropdown(options=[('Felújított (5)', 5), ('Jó állapotú (4)', 4), ('Közepes (3)', 3), ('Felújítandó (2)', 2), ('Új építésű (6)', 6)], value=4, description='Állapot:')
w_panel = widgets.RadioButtons(options=[('Tégla', 0), ('Panel', 1)], value=0, description='Típus:')
w_metro_dist = widgets.IntSlider(min=100, max=2500, step=100, value=600, description='Metró (m):')
w_vasut_dist = widgets.IntSlider(min=50, max=1500, step=50, value=400, description='Vasút (m):')

out_calc = widgets.Output()

def szamol_ertek(*args):
    x_input = pd.DataFrame([{
        'korrigalt_alapterulet_nm': float(w_terulet.value),
        'szobaszam_osszes': 2.0 if w_terulet.value < 60 else 3.0,
        'is_panel': float(w_panel.value),
        'has_lift': 1.0 if w_panel.value == 1 else 0.0,
        'allapot_kod': float(w_allapot.value),
        'tavolsag_metro_halozati_m': float(w_metro_dist.value),
        'tavolsag_vasut_m': float(w_vasut_dist.value),
        'tavolsag_vasut_halozati_m': float(w_vasut_dist.value * 1.2),
        'tavolsag_mazsa_halozati_m': 1000.0
    }])
    pred_nm = rf.predict(x_input)[0]
    pred_tot = (pred_nm * w_terulet.value) / 1e6
    
    with out_calc:
        clear_output(wait=True)
        pred_kpis = [
            ("Becsült Lakásár", fmt_mft(pred_tot), "ML modell előrejelzés", "#10b981"),
            ("Becsült Fajlagos Ár", fmt_huf(pred_nm), "Kínálati becslés", "#2563eb"),
            ("Modell Típus", "Random Forest Regressor", "150 döntési fa", "#7c3aed")
        ]
        display(HTML(kpi_grid_html(pred_kpis)))

w_terulet.observe(szamol_ertek, names='value')
w_allapot.observe(szamol_ertek, names='value')
w_panel.observe(szamol_ertek, names='value')
w_metro_dist.observe(szamol_ertek, names='value')
w_vasut_dist.observe(szamol_ertek, names='value')

display(widgets.VBox([
    widgets.HBox([w_terulet, w_allapot]),
    widgets.HBox([w_panel, w_metro_dist, w_vasut_dist])
]))
display(out_calc)
szamol_ertek()

Output()